In [0]:
# %sql

# -- -- Create a replica of target table to test above migration script
# -- CREATE TABLE default.factrequestcustomcalcfields_test
# -- DEEP CLONE default.factrequestcustomcalcfields;

# -- DROP TABLE IF EXISTS default.factrequestcustomcalcfields_test;
# -- DROP TABLE IF EXISTS default.factrequestcustomcalcfields_backup;


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- Edit these for csv path, target table name and schema -------------------

file_path = f"s3a://{s3_bucket}/edw_exports/{env}/factRequestCustomCalcFields__202603201316.csv"

# target_table_name = "default.factrequestcustomcalcfields_test"
target_table_name = "default.factrequestcustomcalcfields"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("foirequestid", IntegerType(), True),
    StructField("runcycleid", IntegerType(), True),
    StructField("identityverification", StringType(), True),
    StructField("previousfoirequest", StringType(), True),
    StructField("applicationfeepaid", StringType(), True),
    StructField("applicationfeerefunded", StringType(), True),
    StructField("applicationfeetransferred", StringType(), True),
    StructField("linkedrequests", StringType(), True),
    StructField("feepaidamount", StringType(), True),
    StructField("refundamount", StringType(), True),
    StructField("prepaymentamount", StringType(), True),
    StructField("physicalpageestimate", StringType(), True),
    StructField("electronicpageestimate", StringType(), True),
    StructField("customfieldstatus", StringType(), True),
    StructField("secondaryusers", StringType(), True),
    StructField("applicantfilereference", StringType(), True),
    StructField("coordinatednrresponsereqd", StringType(), True),
    StructField("portfolioofficer", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("reviewtype", StringType(), True),
    StructField("judicialreview", StringType(), True),
    StructField("oipcno", StringType(), True),
    StructField("publicationreason", StringType(), True),
    StructField("publication", StringType(), True),
    StructField("crossgovtno", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("currentactivitydate", TimestampType(), True),
    StructField("noofdocdelivered", IntegerType(), True),
    StructField("onholddays", IntegerType(), True),
    StructField("countontime", IntegerType(), True),
    StructField("countoverdue", IntegerType(), True),
    StructField("daysoverdue", IntegerType(), True),
    StructField("passduedays", IntegerType(), True),
    StructField("noofpagesreviewed", IntegerType(), True),
    StructField("noofpagesreleased", IntegerType(), True),
    StructField("noofpagesdeduplicated", IntegerType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(csv_schema)
    .load(file_path)
)

# 3. Add the hardcoded column to the source DataFrame
csv_df_updated = csv_df.withColumns({
    "sourceoftruth": lit("AXIS"),
    "onholdotherdays": lit(None).cast(IntegerType()),
    "processeddays": lit(None).cast(IntegerType()),
    "remainingdays": lit(None).cast(IntegerType()),
    "watcher_name_list": lit(None).cast(StringType())
})

print(f"Number of rows in CSV DataFrame: {csv_df_updated.count()}")

# 4. Upsert (Merge) into the target Delta table
if spark.catalog.tableExists(target_table_name):
    delta_table = DeltaTable.forName(spark, target_table_name)
    
    # merge condition
    merge_condition = """
        target.foirequestid = source.foirequestid AND 
        target.sourceoftruth = source.sourceoftruth
    """
    
    (delta_table.alias("target")
        .merge(
            csv_df_updated.alias("source"),
            merge_condition 
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Migration complete: CSV data upserted.")

else:
    (csv_df_updated.write
        .format("delta")
        .mode("overwrite") 
        .saveAsTable(target_table_name)
    )
    print("Migration complete: New Delta table created from CSV.")